# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Мурашко Владислав'  # напишіть тут своє прізвище та ім'я
ГРУПА = 'ШІ-34'
НОМЕР = 10  # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

with open('lr3_variants_student.json', 'r', encoding='utf-8') as f:
    ВАРІАНТИ = json.load(f)
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 10   ·   Мурашко Владислав   ·   ШІ-34

Ваші вісім стовпців:
   1. Телефон
   2. Супутній
   3. ступінь КТ
   4. Поступлення
   5. КТ
   6. АТ пост
   7. Лім1
   8. Цукор1

Ваша пара для аналізу зв’язку: Вік та помер/жив

Ваше додаткове питання:
   Чому для стовпця «ступінь КТ» медіана має сенс, а середнє — ні? Наведіть конкретний приклад із вашими даними.


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Супутній,ступінь КТ,Поступлення,КТ,АТ пост,Лім1,Цукор1
0,0958469788 дружина Ганна,"ІХС, КС, СН1, НВ ФК2",сердній,2022-02-24,20-30%,140/100,30.0,7.6
1,"0671778486, 0994887663 дружина Оксана","ЦД, ІІ тип",середній,2022-02-22,30-35%,130/80,9.8,19.9
2,"0958768045, чоловік Степан 0951784726","ІХС. КС, ГХІІ, ГЛШ СНІІа, порушення толерантності до глю...",критичний,2021-12-30,70-80%,150/100,12.0,6.2
3,0959819331. дочка Валентина 0960845205,"ЦД, ІІ тип, ІХС, Диф кардіоск, стенокар напруги, синусов...",критичний,2021-12-24,0.8,170/110,8.0,15.5
4,"0952294866, дочка Дарина 0678360161",ІХС ДК ГХІІ-ІІІ СНІІа ГЛШст3,NaN,2022-02-12,NaN,160/100,5.0,4.4
5,0959922983 сестра Софія,"ІХС ДК ГХІІ ГЛШ СНІІ ФК2, ЦД ІІ",середній,2021-12-15,30-40%,140/90,6.0,33.8
6,"0998073702, дружина Наталія 0998382226","ІХС ДК ГХ ІІа ФК3, ЦД вперше виявлений",важкий,2022-04-23,60-65%,130/80,11.0,5.9
7,"0637258771, син Сергій 0968413301","ІХС ДК ГХІІ ГЛШ СНІІа ФК3, полімедик алергія,ерозивий г/...",легкий,2022-05-18,до 17%,140/80,17.0,5.7


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Телефон      120 non-null    object        
 1   Супутній     115 non-null    str           
 2   ступінь КТ   42 non-null     str           
 3   Поступлення  120 non-null    datetime64[us]
 4   КТ           44 non-null     object        
 5   АТ пост      120 non-null    str           
 6   Лім1         117 non-null    float64       
 7   Цукор1       118 non-null    float64       
dtypes: datetime64[us](1), float64(2), object(2), str(3)
memory usage: 7.7+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Супутній
    непорожніх 115 з 121 | унікальних 91 | типи: str×115
    приклади: ["'ІХС, КС, СН1, НВ ФК2'", "'ЦД, ІІ тип'", "'ІХС. КС, ГХІІ, ГЛШ СНІІа, порушення толерантності до глюкози'", "'ЦД, ІІ тип, ІХС, Диф кардіоск, стенокар напруги, синусова тахік, ГХІІ, СНІІв '"]

--- ступінь КТ
    непорожніх 42 з 121 | унікальних 12 | типи: str×42
    приклади: ["'сердній'", "'середній'", "'критичний'", "'важкий'"]

--- Поступлення
    непорожніх 120 з 121 | унікальних 93 | типи: Timestamp×120
    приклади: ["Timestamp('2022-02-24 00:00:00')", "Timestamp('2022-02-22 00:00:00')", "Timestamp('2021-12-30 00:00:00')", "Timestamp('2021-12-24 00:00:00')"]

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    'Телефон': (
        'Ідентифікатор',
        'Номер телефону ідентифікує особу або контакт, але не є вимірюваною величиною.'
    ),
    'Супутній': (
        'Номінальна',
        'Перелік супутніх захворювань описує категорії діагнозів, між якими немає природного порядку.'
    ),
    'ступінь КТ': (
        'Порядкова',
        'Ступінь ураження має природний порядок від середнього до важкого та критичного, але відстані між рівнями невідомі.'
    ),
    'Поступлення': (
        'Інтервальна',
        'Дата госпіталізації дає змогу порівнювати часові інтервали, але нуль календарного відліку умовний.'
    ),
    'КТ': (
        'Відношень',
        'Відсоток ураження легень має природний нуль, а співвідношення значень має зміст.'
    ),
    'АТ пост': (
        'Відношень',
        'Систолічний і діастолічний тиск мають природний нуль, а відношення їхніх значень має фізичний зміст.'
    ),
    'Лім1': (
        'Відношень',
        'Концентрація лімфоцитів має природний нуль, а різниці та відношення значень мають зміст.'
    ),
    'Цукор1': (
        'Відношень',
        'Концентрація глюкози має природний нуль, а різниці та відношення значень мають зміст.'
    )
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,Ідентифікатор,"Номер телефону ідентифікує особу або контакт, але не є в..."
1,Супутній,Номінальна,"Перелік супутніх захворювань описує категорії діагнозів,..."
2,ступінь КТ,Порядкова,Ступінь ураження має природний порядок від середнього до...
3,Поступлення,Інтервальна,Дата госпіталізації дає змогу порівнювати часові інтерва...
4,КТ,Відношень,"Відсоток ураження легень має природний нуль, а співвідно..."
5,АТ пост,Відношень,"Систолічний і діастолічний тиск мають природний нуль, а ..."
6,Лім1,Відношень,"Концентрація лімфоцитів має природний нуль, а різниці та..."
7,Цукор1,Відношень,"Концентрація глюкози має природний нуль, а різниці та ві..."


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
def допустима_статистика(стовпець, шкала):
    з = df[стовпець].dropna()
    ш = шкала.strip().lower()

    р = {
        'непорожніх': len(з),
        'унікальних': з.nunique()
    }

    # Мода та частоти для всіх шкал
    частоти = з.value_counts()
    р['мода'] = частоти.index[0] if len(частоти) else None
    р['частоти'] = частоти.to_dict()

    # Ідентифікатори: лише мода та частоти
    if ш == 'ідентифікатор':
        return р

    # Номінальна шкала: лише мода та частоти
    if ш == 'номінальна':
        return р

    # Порядкова шкала: медіана та квантилі
    if ш == 'порядкова':
        # Для ступеня КТ задаємо природний порядок
        порядок = ['середній', 'важкий', 'критичний']

        категорії = з.astype(str).str.strip().str.lower()
        к = pd.Categorical(
            категорії,
            categories=порядок,
            ordered=True
        )

        ранги = pd.Series(к).dropna().cat.codes

        р['медіана'] = (
            pd.Series(ранги).median()
        )
        р['квантилі'] = {
            '25%': pd.Series(ранги).quantile(0.25),
            '50%': pd.Series(ранги).quantile(0.50),
            '75%': pd.Series(ранги).quantile(0.75)
        }
        return р

    # Інтервальна та вищі шкали
    числові = pd.to_numeric(з, errors='coerce').dropna()

    if len(числові) == 0:
        р['примітка'] = 'Немає придатних числових значень'
        return р

    р['середнє'] = числові.mean()
    р['стандартне відхилення'] = числові.std()

    # Відношень та абсолютна шкали
    if ш in ('відношень', 'абсолютна'):
        середнє = числові.mean()

        if середнє != 0:
            р['коефіцієнт варіації'] = (
                    числові.std() / abs(середнє) * 100
            )
        else:
            р['коефіцієнт варіації'] = None

        р['мінімум'] = числові.min()
        р['максимум'] = числові.max()

    return р


for c in МОЇ:
    print(f'--- {c} ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()


--- Телефон (Ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120, 'мода': '0958469788 дружина Ганна', 'частоти': {'0958469788 дружина Ганна': 1, '0671778486, 0994887663 дружина Оксана': 1, '0958768045, чоловік Степан 0951784726': 1, '0959819331. дочка Валентина 0960845205': 1, '0952294866, дочка Дарина 0678360161': 1, '0959922983 сестра Софія': 1, '0998073702, дружина Наталія 0998382226': 1, '0637258771, син Сергій 0968413301': 1, '0963194893, сестра Катерина 0964948214': 1, '0635244106 син Микола': 1, 504118955: 1, 955939761: 1, 635060454: 1, '0679110285 подруга Зоряна': 1, '0993228088, син Степан 0954230048': 1, 961280845: 1, 990370382: 1, 'син владислав 0997879432': 1, 'син Ярослав 0638421083': 1, '0638440932, дружина Ганна 0634845263': 1, 688280258: 1, 'син Володимир 0679932157': 1, '0675795111, дочка 0954432568': 1, '0966327926, син 0635154742': 1, 'дружина Юлія 0950856517': 1, '0675996775, син Сергій 0500314503': 1, '0953924196, чол.Володимир 0998982104': 1, '0954754072, доч

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Супутній,115,91,ІХС ДК ГХст2,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ступінь КТ,42,12,середній,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Поступлення,120,NaN,NaN,NaN,2022-01-12 07:24:00,2021-02-25 00:00:00,2021-11-05 06:00:00,2022-02-27 12:00:00,2022-04-07 00:00:00,2022-06-03 00:00:00,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
АТ пост,120,23,130/80,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Лім1,117.0,NaN,NaN,NaN,14.281197,1.0,7.0,12.0,18.0,57.0,10.497667
Цукор1,118.0,NaN,NaN,NaN,7.982966,3.2,5.1,6.9,9.0,33.8,4.725707


In [10]:
БЕЗГЛУЗДІ = [
    (
        'Телефон',
        'Середнє',
        df['Телефон'].apply(
            lambda x: pd.to_numeric(x, errors='coerce')
        ).mean(),
        'Номер є ідентифікатором, а не вимірюваною величиною.'
    ),
    (
        'Телефон',
        'Стандартне відхилення',
        df['Телефон'].apply(
            lambda x: pd.to_numeric(x, errors='coerce')
        ).std(),
        'Розкид номерів телефонів нічого не говорить про людей '
        'або їхні властивості: цифри номера не є кількісною шкалою.'
    ),
    (
        'Супутній',
        'Середнє',
        df['Супутній'].astype('category').cat.codes.mean(),
        'Діагнози є номінальними категоріями. Їх кодування числами '
        'не створює кількісної шкали, тому середнє кодів не має сенсу.'
    )
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,Середнє,7.587990e+08,"Номер є ідентифікатором, а не вимірюваною величиною."
1,Телефон,Стандартне відхилення,1.863112e+08,Розкид номерів телефонів нічого не говорить про людей аб...
2,Супутній,Середнє,4.004959e+01,Діагнози є номінальними категоріями. Їх кодування числам...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Вік × помер/жив

Вік: шкала — не у вашому списку
помер/жив: шкала — не у вашому списку

   Вік  помер/жив
0   55          1
1   43          2
2   54          2
3   68          2
4   65          2
5   66          1


In [12]:
ОБРАНА_МІРА = 'точковий бісеріальний коефіцієнт'  # напишіть назву міри
ЧОМУ_САМЕ_ВОНА = 'вік відноситься до шкали відношення, а помер/жив - до бінарної номінальної'  # одне-два речення

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

# ваш код: обчисліть обрану міру
from scipy.stats import pointbiserialr

дані = df[[A, B]].dropna().copy()

# Перетворення статусу на бінарну змінну
дані[B] = дані[B].map({
    2: 0,
    1: 1
})

# Видалення значень, які не вдалося закодувати
дані = дані.dropna()
# Обчислення точково-бісеріального коефіцієнта
r, p = pointbiserialr(
    дані[B],
    дані[A]
)

print('Точково-бісеріальний коефіцієнт:', round(r, 4))
print('p-value:', round(p, 4))
print('Кількість спостережень:', len(дані))

# Інтерпретація
if p < 0.05:
    print('Зв’язок статистично значущий (p < 0.05).')
else:
    print('Статистично значущого зв’язку не виявлено (p >= 0.05).')

if r > 0:
    print('Старший вік пов’язаний із більшою частотою смерті.')
elif r < 0:
    print('Старший вік пов’язаний із меншою частотою смерті.')
else:
    print('Лінійного зв’язку між віком і статусом не виявлено.')

Точково-бісеріальний коефіцієнт: 0.4488
p-value: 0.0
Кількість спостережень: 121
Зв’язок статистично значущий (p < 0.05).
Старший вік пов’язаний із більшою частотою смерті.


In [13]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.
дані = df[['Вік', 'Телефон']].copy()

# Перетворюємо телефон на числовий код категорії
дані['Телефон_код'] = pd.factorize(дані['Телефон'])[0]

# Обчислюємо кореляцію Пірсона
r = дані['Вік'].corr(дані['Телефон_код'], method='pearson')

print('Кореляція Пірсона (Вік × Телефон):', r)


Кореляція Пірсона (Вік × Телефон): 0.36844891803518043


Кореляція Пірсона не повинна застосовуватися механічно до будь-яких числових стовпців. Вона потребує змістовно кількісних змінних. Якщо застосувати її до ідентифікаторів або номінальних категорій, можна отримати числовий результат, який не відображає реального статистичного зв'язку. Число існує математично, але не має змістовної інтерпретації.

---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [14]:
ПІДГОТОВКА = {
    'Телефон': 'Виключити з моделі, оскільки це ідентифікатор пацієнта.',

    'Супутній': 'One-hot кодування, оскільки діагнози є номінальними категоріями без природного порядку.',

    'ступінь КТ': 'Порядкове кодування: середній = 0, важкий = 1, критичний = 2.',

    'Поступлення': 'Перетворити дату на кількість днів від найранішої дати госпіталізації.',

    'КТ': 'Перетворити відсоткові інтервали на числові значення та стандартизувати.',

    'АТ пост': 'Розділити на систолічний і діастолічний тиск та стандартизувати обидва показники.',

    'Лім1': 'Стандартизація за середнім та стандартним відхиленням.',

    'Цукор1': 'Стандартизація або логарифмування за потреби, оскільки це кількісна величина з природним нулем.'
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,Ідентифікатор,"Виключити з моделі, оскільки це ідентифікатор пацієнта."
1,Супутній,Номінальна,"One-hot кодування, оскільки діагнози є номінальними кате..."
2,ступінь КТ,Порядкова,"Порядкове кодування: середній = 0, важкий = 1, критичний..."
3,Поступлення,Інтервальна,Перетворити дату на кількість днів від найранішої дати г...
4,КТ,Відношень,Перетворити відсоткові інтервали на числові значення та ...
5,АТ пост,Відношень,Розділити на систолічний і діастолічний тиск та стандарт...
6,Лім1,Відношень,Стандартизація за середнім та стандартним відхиленням.
7,Цукор1,Відношень,"Стандартизація або логарифмування за потреби, оскільки ц..."


In [15]:
from sklearn.preprocessing import StandardScaler

df_model = df.copy()

# ---------------------------------
# 1. КТ — відсоток ураження легень
# ---------------------------------

def перетворити_КТ(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, float)):
        # Якщо значення записане як частка (0.8 = 80%)
        if 0 <= x <= 1:
            return x * 100
        return x

    x = str(x).strip().replace('%', '').replace(',', '.')

    if '-' in x:
        межі = x.split('-')
        return (float(межі[0]) + float(межі[1])) / 2

    return pd.to_numeric(x, errors='coerce')


df_model['КТ_числове'] = df_model['КТ'].apply(перетворити_КТ)

scaler_кт = StandardScaler()

df_model['КТ_стандартизоване'] = scaler_кт.fit_transform(
    df_model[['КТ_числове']]
)


# ---------------------------------
# 2. АТ пост — артеріальний тиск
# ---------------------------------

# Розділяємо значення виду 140/90 на дві складові

ат = df_model['АТ пост'].astype(str).str.extract(
    r'(\d+(?:[.,]\d+)?)\s*/\s*(\d+(?:[.,]\d+)?)'
)

df_model['АТ_систолічний'] = pd.to_numeric(
    ат[0].str.replace(',', '.', regex=False),
    errors='coerce'
)

df_model['АТ_діастолічний'] = pd.to_numeric(
    ат[1].str.replace(',', '.', regex=False),
    errors='coerce'
)

scaler_ат = StandardScaler()

df_model[['АТ_систолічний_std', 'АТ_діастолічний_std']] = (
    scaler_ат.fit_transform(
        df_model[['АТ_систолічний', 'АТ_діастолічний']]
    )
)


# ---------------------------------
# 3. Поступлення — дата госпіталізації
# ---------------------------------

df_model['Поступлення'] = pd.to_datetime(
    df_model['Поступлення'],
    errors='coerce'
)

найраніша_дата = df_model['Поступлення'].min()

df_model['Поступлення_днів'] = (
    df_model['Поступлення'] - найраніша_дата
).dt.days


# ---------------------------------
# Перевірка результатів
# ---------------------------------

print('КТ:')
print(df_model[['КТ', 'КТ_числове', 'КТ_стандартизоване']].head())

print('\nАртеріальний тиск:')
print(df_model[[
    'АТ пост',
    'АТ_систолічний',
    'АТ_діастолічний',
    'АТ_систолічний_std',
    'АТ_діастолічний_std'
]].head())

print('\nДата госпіталізації:')
print(df_model[[
    'Поступлення',
    'Поступлення_днів'
]].head())


КТ:
       КТ  КТ_числове  КТ_стандартизоване
0  20-30%        25.0           -0.838203
1  30-35%        32.5           -0.491018
2  70-80%        75.0            1.476361
3     0.8        80.0            1.707818
4     NaN         NaN                 NaN

Артеріальний тиск:
   АТ пост  АТ_систолічний  АТ_діастолічний  АТ_систолічний_std  \
0  140/100           140.0            100.0            0.004944   
1   130/80           130.0             80.0           -0.588281   
2  150/100           150.0            100.0            0.598168   
3  170/110           170.0            110.0            1.784616   
4  160/100           160.0            100.0            1.191392   

   АТ_діастолічний_std  
0             1.028305  
1            -0.703577  
2             1.028305  
3             1.894246  
4             1.028305  

Дата госпіталізації:
  Поступлення  Поступлення_днів
0  2022-02-24             364.0
1  2022-02-22             362.0
2  2021-12-30             308.0
3  2021-12-24        

**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [18]:
ВИПРАВЛЕННЯ = [
    (1,'Пошук середнього від бінарної номінальної ознаки','Бінарні номінальні ознаки мають тільки значення 1/0 або так/ні, середнє для них не має ніякого сенсу','рахувати кількість живих/померших пацієнтів та робити відповідні висновки'),
    (2, 'У вибірці представлено 4 статі','Насправді статі всього 2, просто вони мають різні формати запису','стандартизувати формат запису статі'),
    (2, 'Найбільша група — «ж» (56 осіб, 46%).','Насправді в жіночій групі на 19 осіб більше, бо вони віднеслись до групи «жін»','треба просумувати кількість осіб відповідної статі при стандартизації назви'),
    (3, 'Ступінь ураження відсортований за алфавітом, а не за рівнем зростання','Це звичайна поведінка сортування','Кожному ступеню ураження треба надати числовий індекс'),
    (3, 'Багато ступенів ураження мають декілька входжень','Кожен записував ступінь як хотів','Треба стандартизувати назви ступенів'),
    (4, 'Числа неправильні','У таблиці багато значень КТ записані у вигляді діапазонів, вони не враховувались при підрахунках','для таких значень треба взяти середнє значення діапазону'),
    (5, 'Мінімум пораховано неправильно','У таблиці воно записано у відсотках (помилка), що автоматично було представлено як 0.9','прибрати знак відсотку з комірки таблиці'),
    (7, 'Не враховуються всі дані з таблиці статі','як було видно з другого прикладу, там ще є записи "чол" та "жін" - вони не враховані','спочатку стандартизувати стовпець статі'),
    (7, 'Коефіцієнт кореляції Пірсона не підходить для дослідження цих даних','Маємо справу з двома бінарними номінальними ознаками, цей метод для них не підходить','Використати Критерій хі-квадрат Пірсона або Точний тест Фішера'),
    (9, 'Було очищено весь набір даних','В кожному рядку були деякі пропуски, тому вони видалялись','Видаляти тільки повністю порожні рядки, а часткові пропуски заповнити початковими значеннями, або NaN'),
    # (розділ, 'що не так', 'чому число неправильне', 'як правильно'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Пошук середнього від бінарної номінальної ознаки,Бінарні номінальні ознаки мають тільки значення 1/0 або ...,рахувати кількість живих/померших пацієнтів та робити ві...
1,2,У вибірці представлено 4 статі,"Насправді статі всього 2, просто вони мають різні формат...",стандартизувати формат запису статі
2,2,"Найбільша група — «ж» (56 осіб, 46%).","Насправді в жіночій групі на 19 осіб більше, бо вони від...",треба просумувати кількість осіб відповідної статі при с...
3,3,"Ступінь ураження відсортований за алфавітом, а не за рів...",Це звичайна поведінка сортування,Кожному ступеню ураження треба надати числовий індекс
4,3,Багато ступенів ураження мають декілька входжень,Кожен записував ступінь як хотів,Треба стандартизувати назви ступенів
5,4,Числа неправильні,У таблиці багато значень КТ записані у вигляді діапазоні...,для таких значень треба взяти середнє значення діапазону
6,5,Мінімум пораховано неправильно,"У таблиці воно записано у відсотках (помилка), що автома...",прибрати знак відсотку з комірки таблиці
7,7,Не враховуються всі дані з таблиці статі,"як було видно з другого прикладу, там ще є записи ""чол"" ...",спочатку стандартизувати стовпець статі
8,7,Коефіцієнт кореляції Пірсона не підходить для дослідженн...,"Маємо справу з двома бінарними номінальними ознаками, це...",Використати Критерій хі-квадрат Пірсона або Точний тест ...
9,9,Було очищено весь набір даних,"В кожному рядку були деякі пропуски, тому вони видалялись","Видаляти тільки повністю порожні рядки, а часткові пропу..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [19]:
print("-"*20, "НЕПРАВИЛЬНО","-"*20)
стать = df['Стать'].value_counts()
print('У вибірці представлено', df['Стать'].nunique(), 'статі:')
print(стать.to_string())

print('Найбільша група — «ж» (56 осіб, 46%).')
print('Групи «жін» та «чол» помітно менші, що потребує окремого пояснення.')

print("-"*20, "ПРАВИЛЬНО","-"*20)
# Стандартизуємо записи статі
df['Стать_стандартизована'] = (
    df['Стать']
    .astype('string')
    .str.strip()
    .str.lower()
    .replace({
        'ж': 'жінка',
        'жін': 'жінка',
        'жінка': 'жінка',
        'ч': 'чоловік',
        'чол': 'чоловік',
        'чоловік': 'чоловік'
    })
)

# Підраховуємо кількість пацієнтів кожної статі
стать = df['Стать_стандартизована'].value_counts()

print('У вибірці представлено', df['Стать_стандартизована'].nunique(), 'статі:')
print(стать.to_string())

print()
print('Найбільша група —', стать.idxmax(),
      '(', стать.max(), 'осіб,',
      round(стать.max() / стать.sum() * 100, 2), '%).')

print("-"*20, "НЕПРАВИЛЬНО","-"*20)
s = df['spO2 без дот']

print('Сатурація при поступленні без дотації кисню')
print('  середнє      :', round(s.mean(), 2))
print('  медіана      :', round(s.median(), 2))
print('  мінімум      :', round(s.min(), 2))
print('  максимум     :', round(s.max(), 2))

print("-"*20, "ПРАВИЛЬНО","-"*20)
# Перетворюємо значення сатурації на числовий формат
s = pd.to_numeric(
    df['spO2 без дот'].astype('string'),
    errors='coerce'
)

# Виправляємо значення, записані як частки замість відсотків
s = s.where(s > 1, s * 100)

print('Сатурація при поступленні без дотації кисню')
print('  середнє      :', round(s.mean(), 2), '%')
print('  медіана      :', round(s.median(), 2), '%')
print('  мінімум      :', round(s.min(), 2), '%')
print('  максимум     :', round(s.max(), 2), '%')



print("-"*20, "НЕПРАВИЛЬНО","-"*20)
чистий = df.dropna()

print('Було рядків :', len(df))
print('Стало рядків:', len(чистий))
print()
print('Після видалення рядків із пропусками отримано повністю заповнений набір,')
print('придатний для побудови моделі.')


print("-"*20, "ПРАВИЛЬНО","-"*20)
# Видаляємо тільки рядки, у яких усі значення відсутні
чистий = df.dropna(how='all').copy()

print('Було рядків:', len(df))
print('Стало рядків:', len(чистий))
print('Видалено повністю порожніх рядків:', len(df) - len(чистий))
print()

# Перевіряємо кількість пропусків у кожному стовпці
print('Кількість пропусків після очищення:')
print(чистий.isna().sum().to_string())

print()
print('Частково заповнені рядки збережено.')
print('Пропущені значення залишено як NaN для подальшої обробки.')

-------------------- НЕПРАВИЛЬНО --------------------
У вибірці представлено 4 статі:
Стать
ж      56
ч      34
жін    19
чол    12
Найбільша група — «ж» (56 осіб, 46%).
Групи «жін» та «чол» помітно менші, що потребує окремого пояснення.
-------------------- ПРАВИЛЬНО --------------------
У вибірці представлено 2 статі:
Стать_стандартизована
жінка      75
чоловік    46

Найбільша група — жінка ( 75 осіб, 61.98 %).
-------------------- НЕПРАВИЛЬНО --------------------
Сатурація при поступленні без дотації кисню
  середнє      : 82.05
  медіана      : 84.0
  мінімум      : 0.9
  максимум     : 98.0
-------------------- ПРАВИЛЬНО --------------------
Сатурація при поступленні без дотації кисню
  середнє      : 82.79 %
  медіана      : 84.0 %
  мінімум      : 50.0 %
  максимум     : 98.0 %
-------------------- НЕПРАВИЛЬНО --------------------
Було рядків : 121
Стало рядків: 0

Після видалення рядків із пропусками отримано повністю заповнений набір,
придатний для побудови моделі.
----------

---
# Етап 7. Висновки та індивідуальне питання

In [20]:
ВИСНОВКИ = [
    'Набір даних містить інформацію про пацієнтів, зокрема їхні демографічні характеристики, супутні захворювання, результати обстежень, показники сатурації та артеріального тиску. Під час аналізу виявлено різні типи ознак: номінальні, порядкові, інтервальні та кількісні, тому для кожної з них потрібно використовувати відповідні методи статистичної обробки.',

    'Під час дослідження виявлено проблеми з якістю даних: пропущені значення, різні формати запису однакових категорій, числові показники у вигляді тексту, діапазони значень та помилки введення. Для коректного аналізу необхідно стандартизувати категоріальні ознаки, перетворити текстові числові значення у відповідний формат, виправити помилки та зберігати частково заповнені рядки.',

    'Для підготовки даних до побудови моделей необхідно враховувати тип кожної ознаки: категоріальні змінні кодувати, порядковим категоріям призначати числові індекси відповідно до рівня зростання, а кількісні показники перетворювати на числовий формат і за потреби стандартизувати. Кореляційні та статистичні методи потрібно обирати відповідно до типу досліджуваних змінних, щоб отримані результати були коректними та придатними для інтерпретації.'
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Набір даних містить інформацію про пацієнтів, зокрема їхні демографічні характеристики, супутні захворювання, результати обстежень, показники сатурації та артеріального тиску. Під час аналізу виявлено різні типи ознак: номінальні, порядкові, інтервальні та кількісні, тому для кожної з них потрібно використовувати відповідні методи статистичної обробки.
2. Під час дослідження виявлено проблеми з якістю даних: пропущені значення, різні формати запису однакових категорій, числові показники у вигляді тексту, діапазони значень та помилки введення. Для коректного аналізу необхідно стандартизувати категоріальні ознаки, перетворити текстові числові значення у відповідний формат, виправити помилки та зберігати частково заповнені рядки.
3. Для підготовки даних до побудови моделей необхідно враховувати тип кожної ознаки: категоріальні змінні кодувати, порядковим категоріям призначати числові індекси відповідно до рівня зростання, а кількісні показники перетворювати на числовий формат і за потр

In [21]:
print('ВАШЕ ПИТАННЯ:');
print(В['питання']);
print()
ВІДПОВІДЬ = 'Стовпець «ступінь КТ» є порядковою ознакою, оскільки його значення можна розташувати за рівнем тяжкості ураження: від меншого до більшого. Але відстань між сусідніми категоріями не є однаковою або точно вимірюваною. Медіана показує категорію, яка знаходиться посередині впорядкованого ряду спостережень. Для її визначення достатньо знати порядок категорій, а не числову відстань між ними. Для обчислення середнього потрібно надати категоріям числові значення. Але якщо умовно присвоїти категоріям індекси 1, 2, 3, 4, то середнє цих індексів залежатиме від обраного кодування, а не від реальної тяжкості ураження. Припустимо, що у нас 3 пацієнти зі ступенями ураження Легкий (1), Важкий (3), Критичний (4). Медіаною буде важкий стан, а середнім значенням - 2.67. Число 2.67 не має відповідної категорії, тому й пошук середнього не має сенсу.'
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Чому для стовпця «ступінь КТ» медіана має сенс, а середнє — ні? Наведіть конкретний приклад із вашими даними.

Стовпець «ступінь КТ» є порядковою ознакою, оскільки його значення можна розташувати за рівнем тяжкості ураження: від меншого до більшого. Але відстань між сусідніми категоріями не є однаковою або точно вимірюваною. Медіана показує категорію, яка знаходиться посередині впорядкованого ряду спостережень. Для її визначення достатньо знати порядок категорій, а не числову відстань між ними. Для обчислення середнього потрібно надати категоріям числові значення. Але якщо умовно присвоїти категоріям індекси 1, 2, 3, 4, то середнє цих індексів залежатиме від обраного кодування, а не від реальної тяжкості ураження. Припустимо, що у нас 3 пацієнти зі ступенями ураження Легкий (1), Важкий (3), Критичний (4). Медіаною буде важкий стан, а середнім значенням - 2.67. Число 2.67 не має відповідної категорії, тому й пошук середнього не має сенсу.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [22]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних': 'текст, перероблено',
    'етап 2 — допустимі статистики': 'код, перевірено',
    'етап 3 — безглузді числа': 'код, перевірено',
    'етап 4 — міра зв’язку': 'код, перевірено',
    'етап 5 — підготовка до моделі': 'код, перевірено',
    'етап 6 — виправлення аналізу': 'код, перевірено',
    'етап 7 — висновки й відповідь': 'текст, перероблено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Перевірив правильність виконання коду та його відповідність поставленій задачі (чи враховує він всі необхідні нюанси), за необхідністю виправляв. Текст перевірено на відповідність реальним даним та висновкам, майже всі тексти дописувались та редагувались.'
ПІДТВЕРДЖУЮ = True  # я можу пояснити кожен рядок свого коду
# і внести в нього зміни на вимогу під час захисту

In [23]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля': bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців': all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа': len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано': all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок': len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки': len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання': len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено': all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів': len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства': ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 10 готова до захисту.
Мурашко Владислав, ШІ-34


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.